In [ ]:
from math import ceil, floor
from pathlib import Path
import geopandas as gpd

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin

lc_guatemala = gpd.read_file(r'C:\Users\Paula_Copelli\Documents\COPERNICUS\TALLERES\03_GUATEMALA_2026\data\shapes\landcover_guatemala_2020.shp')

resolucion = 20
xmin, ymin, xmax, ymax = lc_guatemala.total_bounds
xmin = floor(xmin / resolucion) * resolucion
xmax = ceil(xmax / resolucion) * resolucion
ymin = floor(ymin / resolucion) * resolucion
ymax = ceil(ymax / resolucion) * resolucion
transform = from_origin(xmin, ymax, resolucion, resolucion)
width = int((xmax - xmin) / resolucion)
height = int((ymax - ymin) / resolucion)
lc_rasterizar = lc_guatemala.dropna(subset=["C_USO"])

raster = rasterize(
    ((geom, int(codigo)) for geom, codigo in zip(
        lc_rasterizar.geometry,
        lc_rasterizar["C_USO"],
    )),
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="int32",
)

ruta_raster = Path(r"C:\Users\Paula_Copelli\Documents\COPERNICUS\TALLERES\03_GUATEMALA_2026\data\raster\landcover_guatemala_2020_20m.tif")
with rasterio.open(
    ruta_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="int32",
    crs=lc_guatemala.crs,
    transform=transform,
    nodata=0,
) as dst:
    dst.write(raster, 1)

print(f"Raster guardado en: {ruta_raster}")


Raster guardado en: C:\Users\Paula_Copelli\Documents\COPERNICUS\TALLERES\03_GUATEMALA_2026\data\raster\landcover_guatemala_2020_20m.tif
